In [ ]:
# 1. 필요한 라이브러리를 불러온다.
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
# 2. 실습에 사용할 LLM과 토크나이저를 불러온다.
model_name = "beomi/kcbert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 627.04it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

In [ ]:
# 3. 데이터를 생성하고 BERT 모델이 처리할 수 있는 형태로 토크나이징한다.
texts = [
    # 긍정 데이터
    "전체적인 분위기가 좋아서 편하게 볼 수 있었어요.",
    "스토리는 평범했지만 연출 덕분에 재미있었어요.",
    "배우들의 연기가 자연스러워서 몰입이 잘 됐어요.",
    "큰 기대 없이 봤는데 생각보다 괜찮았어요.",
    "잔잔하지만 끝나고 나서 여운이 남는 영화였어요.",

    # 부정 데이터
    "이야기가 늘어져서 중간부터 집중이 안 됐어요.",
    "연출이 과해서 오히려 몰입을 방해했어요.",
    "캐릭터 행동이 이해되지 않아서 답답했어요.",
    "분위기는 잡으려는 것 같은데 내용이 부족했어요.",
    "전체적으로 뭔가 아쉬운 느낌이 많이 남았어요."
]

inputs = tokenizer(
    texts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=64
)

In [ ]:
# 4. 토크나이징한 결과를 확인한다.
print(inputs)

{'input_ids': tensor([[    2, 10508,  8097, 12655,  4009, 13492, 12944,  1576,  1931,  8429,
          8186,    17,     3,     0,     0,     0],
        [    2, 17319,  8086, 13732, 12075, 23889, 11725, 10827,  4188, 16849,
            17,     3,     0,     0,     0,     0],
        [    2, 10631,  8089, 11219,  4009, 10459, 15152,  4072,  1386,  4307,
          4017,  2483,   896,  8186,    17,     3],
        [    2,  3089,  9298,  8629, 14738, 17580,  9251, 17436,  4040,    17,
             3,     0,     0,     0,     0,     0],
        [    2,  2479,  4571,  9082, 14281,  9005,  2269, 22368, 21049,  9376,
         13198,  4040,    17,     3,     0,     0],
        [    2, 23206,  9986,  9331, 12793,  8042, 11997,  4017,  2173,   896,
          8186,    17,     3,     0,     0,     0],
        [    2, 23889,  4017,   321,  7987,  8937,  1386, 29436, 11366, 13419,
            17,     3,     0,     0,     0,     0],
        [    2,  2996, 27504, 13741,  8226,  9441, 18766,  9224, 1341

In [7]:
print(inputs.input_ids)

tensor([[    2, 10508,  8097, 12655,  4009, 13492, 12944,  1576,  1931,  8429,
          8186,    17,     3,     0,     0,     0],
        [    2, 17319,  8086, 13732, 12075, 23889, 11725, 10827,  4188, 16849,
            17,     3,     0,     0,     0,     0],
        [    2, 10631,  8089, 11219,  4009, 10459, 15152,  4072,  1386,  4307,
          4017,  2483,   896,  8186,    17,     3],
        [    2,  3089,  9298,  8629, 14738, 17580,  9251, 17436,  4040,    17,
             3,     0,     0,     0,     0,     0],
        [    2,  2479,  4571,  9082, 14281,  9005,  2269, 22368, 21049,  9376,
         13198,  4040,    17,     3,     0,     0],
        [    2, 23206,  9986,  9331, 12793,  8042, 11997,  4017,  2173,   896,
          8186,    17,     3,     0,     0,     0],
        [    2, 23889,  4017,   321,  7987,  8937,  1386, 29436, 11366, 13419,
            17,     3,     0,     0,     0,     0],
        [    2,  2996, 27504, 13741,  8226,  9441, 18766,  9224, 13419,    17,
    

In [8]:
print(f"[CLS] 토큰 ID: {tokenizer.cls_token_id}")
print(f"[SEP] 토큰 ID: {tokenizer.sep_token_id}")
print(f"[PAD] 토큰 ID: {tokenizer.pad_token_id}")

[CLS] 토큰 ID: 2
[SEP] 토큰 ID: 3
[PAD] 토큰 ID: 0


In [9]:
print(inputs.token_type_ids)

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])


In [10]:
print(inputs.attention_mask)

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]])


In [14]:
# 5. 앞서 준비한 데이터를 활용해 BERT 모델을 실행한다.
model.eval()
with torch.no_grad():
    outputs = model(**inputs)
    
logits = outputs.logits
predicts = torch.argmax(logits, dim=1)

print(logits)
print(predicts)

tensor([[ 0.2193,  0.1471],
        [ 0.3207, -0.1013],
        [ 0.4944, -0.3338],
        [ 0.2698,  0.1381],
        [ 0.5869, -0.5558],
        [ 0.3045, -0.1468],
        [ 0.4048, -0.0948],
        [ 0.3022, -0.2809],
        [ 0.3935, -0.2880],
        [ 0.4214, -0.3599]])
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])


In [ ]:
# 6. 허깅페이스에서는 기본적으로 0은 부정, 1은 긍정으로 나타낸다.
for pred in predicts:
    if pred == 1:
        result = "긍정"
    else:''
        result = "부정"
    print("감성 분석 결과:", result)

감성 분석 결과: 부정
감성 분석 결과: 부정
감성 분석 결과: 부정
감성 분석 결과: 부정
감성 분석 결과: 부정
감성 분석 결과: 부정
감성 분석 결과: 부정
감성 분석 결과: 부정
감성 분석 결과: 부정
감성 분석 결과: 부정
